# 01 — Prepare the geographic analysis inputs

Connect notebook 00's daily sample to the official district geography. Follow four geographic steps: validated district-to-province mapping, distance reference points, derived province geometries, and a trip-weighted filtering audit. The final section saves the input provenance required by notebook 02.

Keep all 52 province-level units, including the islands, Ceuta, and Melilla. Use the package's validated correspondence rather than inferring province membership from a district identifier.

## Setup

Load the environment and check that notebook 00 has prepared the four study weeks.

In [1]:
import hashlib
import json
import os
import shutil
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import polars as pl
import pyspainmobility as psm
from IPython.display import display
from shapely import make_valid

if not hasattr(psm, "select_od"):
    raise ImportError("Use the current pySpainMobility selection/network API.")

pd.set_option("display.max_colwidth", None)
if psm.__version__ != "2.1.2":
    raise ImportError("This paper workflow requires pySpainMobility 2.1.2; install requirements.txt.")


In [2]:
notebook_names = ["00_download_and_build_graphs.ipynb", "01_prepare_analysis_inputs.ipynb", "02_main_paper_analysis.ipynb"]
configured_paper = os.environ.get("PYSPAIN_PAPER_DIR")
if configured_paper:
    PAPER_DIR = Path(configured_paper).expanduser().resolve()
else:
    PAPER_DIR = next(
        (candidate for parent in [Path.cwd(), *Path.cwd().parents]
         for candidate in [parent, parent / "examples/paper"]
         if all((candidate / name).exists() for name in notebook_names)), None,
    )
    if PAPER_DIR is None:
        raise FileNotFoundError("Open these notebooks from examples/paper or set PYSPAIN_PAPER_DIR.")

PIPELINE_OUTPUT = Path(os.environ.get("PYSPAIN_PIPELINE_OUTPUT", str(PAPER_DIR / "outputs/paper_inputs"))).expanduser().resolve()
RAW_DATA_DIR = Path(os.environ.get("PYSPAIN_RAW_DATA_DIR", str(PAPER_DIR / "raw_data"))).expanduser().resolve()
WINDOWS = {
    "winter": ("2023-01-16", "2023-01-22"),
    "spring": ("2023-05-08", "2023-05-14"),
    "summer": ("2023-07-17", "2023-07-23"),
    "autumn": ("2023-10-09", "2023-10-15"),
}
MAIN = list(WINDOWS)
BANDS = {"<10": "low", "10-15": "mid", ">15": "high"}
GROUPS = list(BANDS.values())
PROVINCES = [f"{number:02d}" for number in range(1, 53)]
print("Notebook directory:", PAPER_DIR)
print("Prepared inputs:", PIPELINE_OUTPUT)

def digest(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()

provenance_path = PIPELINE_OUTPUT / "provenance.json"
if not provenance_path.exists():
    raise FileNotFoundError("Run notebook 00 before preparing geography.")
provenance = json.loads(provenance_path.read_text())
assert provenance["inputs_status"] in {"acquired", "prepared"}
assert set(provenance["windows"]) == set(WINDOWS)
for season, window in WINDOWS.items():
    assert tuple(provenance["windows"][season]) == window
ZONE_DIR = PIPELINE_OUTPUT / "zones"
ZONE_DIR.mkdir(parents=True, exist_ok=True)


def show_summary(title, table, decimals=2):
    print(title)
    display(table.round(decimals))

show_summary("Study weeks available for geographic preparation", pd.DataFrame.from_dict(
    WINDOWS, orient="index", columns=["Start", "End"]
))
print("Input status:", provenance["inputs_status"])


Notebook directory: .
Prepared inputs: outputs/paper_inputs
Study weeks available for geographic preparation


,Start,End
winter,2023-01-16,2023-01-22
spring,2023-05-08,2023-05-14
summer,2023-07-17,2023-07-23
autumn,2023-10-09,2023-10-15


Input status: acquired


## 1. District geography and validated correspondence

Reuse local official geography files when present; otherwise the Zones API obtains them. Exclude a source district only when it lacks a unique valid province correspondence. Record these exclusions rather than assigning a province from its identifier prefix.

In [3]:
source_files = ["distritos_2.geojson", "relacion_ine_zonificacionMitma.csv", "nombres_distritos.csv"]
for filename in source_files:
    local = RAW_DATA_DIR / filename
    target = ZONE_DIR / filename
    if local.exists() and (not target.exists() or digest(target) != digest(local)):
        shutil.copy2(local, target)
zones = psm.Zones(zones="districts", version=2, output_directory=str(ZONE_DIR))
all_districts = zones.get_zone_geodataframe()
assert all_districts.crs is not None and all_districts.index.is_unique
mapping_frame = zones.get_province_mapping(source_ids=all_districts.index, unmapped="exclude")
assert mapping_frame.source_id.is_unique
assert set(mapping_frame.target_id) == set(PROVINCES)
excluded = sorted(mapping_frame.attrs.get("unmapped_source_ids", []))
mapping_frame.to_csv(ZONE_DIR / "district_to_province.csv", index=False)

districts = all_districts.loc[all_districts.index.isin(mapping_frame.source_id)].copy()
districts.geometry = districts.geometry.map(make_valid)
assert len(districts) == len(mapping_frame)
print(f"Raw geometries: {len(all_districts):,}; mapped districts: {len(districts):,}; excluded: {len(excluded)}.")


File outputs/paper_inputs/zones/distritos_2.geojson already exists. Loading it...


Raw geometries: 3,792; mapped districts: 3,743; excluded: 49.


In [4]:
show_summary("Example validated district-to-province correspondences", mapping_frame.head(8).rename(
    columns={"source_id": "District ID", "target_id": "Province code"}
))
if excluded:
    show_summary("Examples of geometry IDs without a valid province correspondence", pd.DataFrame(
        {"Excluded district ID": excluded[:8]}
    ))
print("Mapping uniqueness and coverage of all 52 province-level units: passed.")


Example validated district-to-province correspondences


,District ID,Province code
0,01001,01
1,01002,01
2,01004_AM,01
3,01009_AM,01
4,01010,01
5,01017_AM,01
6,01028_AM,01
7,01036,01


Examples of geometry IDs without a valid province correspondence


,Excluded district ID
0,34022253049
1,34024453076
2,34024453077
3,34063953072
4,34070953004
5,34070953005
6,34070953006
7,34070953007


Mapping uniqueness and coverage of all 52 province-level units: passed.


## 2. Representative points for endpoint distances

Repair district geometries and calculate representative points in EPSG:25830. Coordinates are in metres; notebook 02 uses their Euclidean separation, converted to kilometres, as the endpoint-distance measure. This measure does not represent travelled route length.

In [5]:
points = districts.to_crs(25830).representative_point()
district_reference = pd.DataFrame({"district_id": districts.index.astype(str),
                                    "x": points.x.to_numpy(), "y": points.y.to_numpy()})
district_reference = district_reference.merge(mapping_frame, left_on="district_id", right_on="source_id", validate="one_to_one")
district_reference = district_reference.rename(columns={"target_id": "province_code"}).drop(columns="source_id")
if "name" in districts:
    district_reference["district_name"] = district_reference.district_id.map(districts["name"])
if "population" in districts:
    district_reference["population"] = district_reference.district_id.map(districts["population"])
district_reference.to_parquet(ZONE_DIR / "district_reference.parquet", index=False)


assert np.isfinite(district_reference[["x", "y"]].to_numpy()).all()
show_summary("Example distance-reference rows", district_reference.head(6).rename(columns={
    "district_id": "District ID", "province_code": "Province code", "x": "X (m)", "y": "Y (m)",
}))
print(f"Prepared {len(district_reference):,} district reference points; all coordinates are finite.")


Example distance-reference rows


,District ID,X (m),Y (m),Province code,district_name,population
0,01001,537672.80,4739245.00,01,Alegría-Dulantzi,2925.0
1,01002,503981.03,4765558.06,01,Amurrio,10307.0
2,01004_AM,488487.25,4775147.07,01,Artziniega agregacion de municipios,3005.0
3,01009_AM,542235.76,4747624.96,01,Asparrena agregacion de municipios,4599.0
4,01010,493582.93,4769343.93,01,Ayala/Aiara,2951.0
5,01017_AM,547784.77,4724970.76,01,Campezo/Kanpezu agregacion de municipios,4314.0


Prepared 3,743 district reference points; all coordinates are finite.


## 3. Derived province geometries

Use `Zones(zones="provinces")` to aggregate the mapped MITMA district geometries into province-level products. These are derived geometries; they are not an independently published province OD dataset. The reference table records how many mapped districts belong to each province.

In [6]:
province_zones = psm.Zones(zones="provinces", version=2, output_directory=str(ZONE_DIR))
province_geo = province_zones.get_zone_geodataframe()
assert set(province_geo.index.astype(str)) == set(PROVINCES)
province_geo.to_file(ZONE_DIR / "provinces_derived.geojson", driver="GeoJSON")
province_reference = district_reference.groupby("province_code", as_index=False).agg(n_districts=("district_id", "nunique"))
province_reference.to_csv(ZONE_DIR / "province_reference.csv", index=False)

show_summary("Example province-reference rows", province_reference.head(10).rename(columns={
    "province_code": "Province code", "n_districts": "Mapped districts",
}))
print(f"Prepared {len(province_geo)} province geometries from {province_reference.n_districts.sum():,} mapped districts.")
print("Full province-reference table:", ZONE_DIR / "province_reference.csv")


File outputs/paper_inputs/zones/distritos_2.geojson already exists. Loading it...


../../pyspainmobility/zones/zones.py:339: RuntimeWarning: Province geometries exclude 49 unmapped source zones; see result.attrs.
  self._load_zone_geodataframe()


Example province-reference rows


,Province code,Mapped districts
0,01,21
1,02,40
2,03,154
3,04,58
4,05,25
5,06,93
6,07,92
7,08,322
8,09,35
9,10,51


Prepared 52 province geometries from 3,743 mapped districts.
Full province-reference table: outputs/paper_inputs/zones/province_reference.csv


## 4. Geographic filtering audit

The geographic audit separates unmapped endpoints, same-district trips, within-province trips between districts, and inter-province trips. Its four components must sum to the complete-demographic source weight. Nothing is excluded simply because it belongs to an island province.

In [7]:
mapping = dict(zip(mapping_frame.source_id, mapping_frame.target_id))
coverage_rows = []
for season in MAIN:
    paths = [str(PIPELINE_OUTPUT / "cache" / Path(record["cache"]).name)
             for record in provenance["sources"] if record["window"] == season]
    assert len(paths) == 7
    query = pl.scan_parquet(paths).with_columns(
        pl.col("id_origin").replace_strict(mapping, default=None).alias("op"),
        pl.col("id_destination").replace_strict(mapping, default=None).alias("dp"))
    known = pl.col("op").is_not_null() & pl.col("dp").is_not_null()
    same_district = pl.col("id_origin") == pl.col("id_destination")
    same_province = pl.col("op") == pl.col("dp")
    weights = query.select(
        pl.col("n_trips").sum().alias("sample_weight"),
        pl.col("n_trips").filter(~known).sum().alias("unmapped_weight"),
        pl.col("n_trips").filter(known & same_district).sum().alias("district_self_loop_weight"),
        pl.col("n_trips").filter(known & ~same_district & same_province).sum().alias("within_province_interdistrict_weight"),
        pl.col("n_trips").filter(known & ~same_province).sum().alias("interprovince_weight"),
    ).collect().to_dicts()[0]
    np.testing.assert_allclose(sum(value for key, value in weights.items() if key != "sample_weight"), weights["sample_weight"], rtol=1e-9)
    coverage_rows.append({"season": season, **weights})
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(ZONE_DIR / "geographic_flow_accounting.csv", index=False)

stages = {
    "sample_weight": "Complete-demographic sample",
    "unmapped_weight": "Excluded: unmatched endpoints",
    "district_self_loop_weight": "Excluded: same-district trips",
    "within_province_interdistrict_weight": "Excluded: within-province interdistrict trips",
    "interprovince_weight": "Retained: inter-province trips",
}
coverage_display_rows = []
for row in coverage.to_dict("records"):
    for key, label in stages.items():
        coverage_display_rows.append({
            "Season": row["season"].title(), "Stage": label,
            "Expanded trips (million)": row[key] / 1e6,
            "Share of initial sample (%)": 100 * row[key] / row["sample_weight"],
        })
show_summary("Geographic filtering and retained analytical weight", pd.DataFrame(coverage_display_rows))
print("Excluded and retained trip weights sum to the initial sample in each week: passed.")


Geographic filtering and retained analytical weight


,Season,Stage,Expanded trips (million),Share of initial sample (%)
0,Winter,Complete-demographic sample,564.68,100.00
1,Winter,Excluded: unmatched endpoints,1.18,0.21
2,Winter,Excluded: same-district trips,178.68,31.64
3,Winter,Excluded: within-province interdistrict trips,373.30,66.11
4,Winter,Retained: inter-province trips,11.53,2.04
5,Spring,Complete-demographic sample,589.00,100.00
6,Spring,Excluded: unmatched endpoints,1.03,0.18
7,Spring,Excluded: same-district trips,187.55,31.84
8,Spring,Excluded: within-province interdistrict trips,386.70,65.65
9,Spring,Retained: inter-province trips,13.72,2.33


Excluded and retained trip weights sum to the initial sample in each week: passed.


## 5. Complete inputs for notebook 02

Record source-file checksums, the mapping and derived-geometry checksums, unmatched geometry IDs, and the distance method. The input status becomes `prepared` once all geographic products and the flow audit are saved.

Notebook 02 reads the daily OD caches and these geographic inputs to compute the observed results, the residence-conditioned benchmark, and the district–province comparisons.

In [8]:
provenance.update(
    zoning_sources={filename: digest(ZONE_DIR / filename) for filename in source_files},
    mapping_sha256=digest(ZONE_DIR / "district_to_province.csv"),
    province_geometry_sha256=digest(ZONE_DIR / "provinces_derived.geojson"),
    district_universes={"historical_geometry": len(all_districts), "current_province_mapping": len(districts)},
    unmapped_zone_ids=excluded, notebook01_sha256=digest(PAPER_DIR / "01_prepare_analysis_inputs.ipynb"),
    inputs_status="prepared",
)
provenance_path.write_text(json.dumps(provenance, indent=2))

show_summary("Geographic inputs prepared for analysis", pd.DataFrame([
    {"Output": "Validated mapping", "Rows / units": len(mapping_frame), "File": "district_to_province.csv"},
    {"Output": "District distance references", "Rows / units": len(district_reference), "File": "district_reference.parquet"},
    {"Output": "Derived province geometries", "Rows / units": len(province_geo), "File": "provinces_derived.geojson"},
    {"Output": "Province reference table", "Rows / units": len(province_reference), "File": "province_reference.csv"},
    {"Output": "Geographic flow audit", "Rows / units": len(coverage), "File": "geographic_flow_accounting.csv"},
]))
print("Geographic files:", ZONE_DIR)
print("Input status:", provenance["inputs_status"])
print("Next: run 02_main_paper_analysis.ipynb.")


Geographic inputs prepared for analysis


,Output,Rows / units,File
0,Validated mapping,3743,district_to_province.csv
1,District distance references,3743,district_reference.parquet
2,Derived province geometries,52,provinces_derived.geojson
3,Province reference table,52,province_reference.csv
4,Geographic flow audit,4,geographic_flow_accounting.csv


Geographic files: outputs/paper_inputs/zones
Input status: prepared
Next: run 02_main_paper_analysis.ipynb.
